In [ ]:
# 1. サンプルデータセットをダウンロード
%cd /content
!git clone https://github.com/wal-afk/drive_sim
%cd drive_sim
!git pull
!git restore .
!git clean -fd
%cd /content/drive_sim

!pip install -U plotly==6.9

In [1]:
import yaml

from sim.drive_simulator import CarSim
from sim.vehicle import VehicleProp
from sim.mission_base import MissionBase
from sim.goal import GoalLine, GoalCircle
from sim.drawer import SimDrawer, MissionDrawer
from sim.worlds.type_b_world import type_b_circuit
from sim.sign import Sign

with open("config/type-b.yaml", "r") as f:
    vehicle_config = yaml.safe_load(f)

prop = VehicleProp(**vehicle_config)

/usr/lib/python3/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# ミッション7

ロボットをコースに沿って走らせ、1つ目の標識(標識名はoriginal)を踏まないようコース外に避けてからコース内に復帰し、2つ目の標識(標識名はwarn)の0.4m手前で止まろう。
- 標識は２つともコース上に置かれている
- コースに沿って走らせるにはauto命令を使えるが、auto命令は車がコースの中にいて、かつコースの向きと車の向きがあっているときのみ使えることに注意しよう（コースに対して斜めの状態でONにしても上手く走れない）
- シミュレータ上は、1つ目の標識のコース内側（標識の左側）を通って、2つ目の標識の0.4m手前で停止したら成功とする

## 取り組み方
1. 使える命令を理解する
2. 下のセルを実行して、ロボットの限界速度や、ロボットが存在する初期位置を把握する
3. ２つ下のセル内にプログラムを書き実行して結果を見る
4. シミュレータでうまく動いたらロボットにプログラムを書きこんで動かしてみよう（講師に声をかけてね）

|使える命令|意味|指定できる値|使い方の例|
|--|--|--|--|
|move|一定速度で前に進む|v=速度[m/秒]|move(v=0.2)|
|move|一定時間だけ一定速度で前に進む|v=速度[m/秒], t=時間[秒]|move(v=0.2, t=1.0)|
|move|一定時間だけ円弧を描いて進む。r>0は左カーブ、r<0は右カーブ|v=速度[m/秒], r=回転半径[m], t=時間[秒]|move(v=0.2, r=0.5, t=1.0)|
|rotate|一定速度で回転する。w>0は左回転。w<0は右回転|w=回転速度[度/秒]|rotate(w=90)|
|rotate|一定時間だけ一定速度で回転する。w>0は左回転。w<0は右回転|w=回転速度[度/秒], t=時間[秒]|rotate(w=90, t=1.0)|
|wait|直前の命令が終わるまで待つ||wait()|
|serach|標識を見つける（複数見つかった場合は、最も近いもの）||pos = search()|
|serach|特定の標識を見つける（複数見つかった場合は、最も近いもの）|name = "標識名"|pos = search(name="original")|
|auto|コースに沿って進む|v=速度[m/秒]|auto(v=0.2)|

- pos = search() が返す値には下記が含まれる
  - pos.x: 見つけた標識の前方位置[m]　※前方が正
  - pos.y: 見つけた標識の左右位置[m]　※左側が正、右側は負
  - pos.r: 見つけた標識への距離[m]
  - pos.theta: 見つけた標識の角度[度]　※左側が正、右側は負
  - pos.name: 見つけた標識の標識名

## 注意点
- スタート時の位置はランダムに最大10cmほどずれる
- スタート時の向きはランダムに最大5度ほどずれる

In [2]:
class Mission7Base(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=40)
        # シミュレータでは「1つ目の標識のコース内側を通過 → 2つ目の標識の0.4m手前で停止」したら成功とする
        self.goals = [
            GoalCircle((1.6, 0.4), 0.15, should_stop=False),
            GoalCircle((4.07, 0.45), 0.15, should_stop=True),
        ]
        self.initial_xy = (0.2, 0.0)
        self.random_d_xy = (0.1, 0.1)
        self.random_d_yaw_deg = 5
        self.set_signs(
            [
                Sign(x=1.6, y=0.0, name="original"),
                Sign(x=4.25, y=0.81, name="warn"),
            ]
        )


print("最大速度", prop.max_velocity, "m/秒")
print("最大回転速度", prop.max_rotate_deg, "度/秒")
MissionDrawer(Mission7Base()).show()

最大速度 0.22 m/秒
最大回転速度 162.72 度/秒


In [ ]:
class Mission7(Mission7Base):
    @staticmethod
    def command_func(alive, *, move, rotate, wait, search, auto, **kwargs):
        # ヒントとして、「searchして標識が見つからなかったらコースに沿って進む」を繰り返すという処理を記載済み
        # 「標識(original)が見つかった場合」と「original以外の標識が見つかった場合」のプログラムを書こう

        while alive():
            pos = search()
            if pos is None:
                auto(v=0.2)
            elif pos.name == "original":
                # ####### ここから下に「標識(original)が見つかった場合」のプログラムを書こう
                auto(v=0.2)
                # ####### ここより上にプログラムを書こう
            else:
                # ####### ここから下に「original以外の標識が見つかった場合」のプログラムを書こう
                auto(v=0.2)
                # ####### ここより上にプログラムを書こう
        # ####### プログラムを書いた後にセルを実行し結果を確認しよう


sim = CarSim(prop, Mission7())
sim.run()
SimDrawer(sim).show()

drive_dt=0.031, detect_dt=0.061, throttle=20


/home/la000017/data/git/drive_sim/sim/drive_simulator.py:579: RuntimeWarning:

invalid value encountered in divide

/home/la000017/data/git/drive_sim/sim/drive_simulator.py:585: RuntimeWarning:

invalid value encountered in divide



!!!!!! time limit 40s: stop simulation
[40.007] simulation_func finished
    takes 1.270s
[40.007] command_func finished
Trajectory points : 1303
Active threads : 7


100%|██████████| 435/435 [00:01<00:00, 234.20it/s]


# ミッション7のヒント
- 動きを次の４段階に分けて考えるとよい
  1. autoでコースを走る
  2. 1つ目の標識(original)の手前まで来たら、標識を踏まないようによける
  3. よけ終わったら、またautoでコースを走る
  4. 2つ目の標識(warn)の0.4m手前まで来たら停止する
- 標識までの前方位置はpos.xなので、「pos.xが0.4より大きいうちはautoで進む」という条件分岐を使おう（ミッション3を思い出そう）
- 1つ目の標識のよけ方の例
  - 標識の0.4m手前で、まず標識の方を向く（ミッション1と同じ方法）
  - 左に90度回転してから、半径0.4mの右カーブで半円を描いて進み、もう一度左に90度回転するとコースに戻れる
  - 円弧を描いて進むには move(v=速度, r=回転半径, t=時間) を使う。半円の長さは「3.14 × 半径」なので、時間は「3.14 × 半径 ÷ 速度」になる
  - それぞれの動きの後には wait() で動きが終わるのを待とう